<a href="https://colab.research.google.com/github/Syamala-darapuneni44/ATF-03/blob/main/resnetdataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [13]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# Set the path to the dataset
dataset_path = '/content/drive/MyDrive/Animal'

# Define the data transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# Load the dataset
dataset = datasets.ImageFolder(dataset_path, data_transforms['train'])

# Create data loaders for training and validation
batch_size = 32
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Define the ResNet model
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights=models.ResNet18_Weights.DEFAULT)

# Freeze the weights of the pre-trained model
for param in model.parameters():
    param.requires_grad = False

# Add a new classification layer
num_classes = len(dataset.classes)  # Get the number of classes from the dataset
model.fc = nn.Linear(512, num_classes)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Train the model
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print('Epoch %d, Loss: %.4f' % (epoch+1, running_loss/(i+1)))

# Evaluate the model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in train_loader:
        inputs, labels = data
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print('Accuracy: %.4f' % accuracy)


Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0


Epoch 1, Loss: 1.4253
Epoch 2, Loss: 1.4032
Epoch 3, Loss: 1.3729
Epoch 4, Loss: 1.2517
Epoch 5, Loss: 1.1336
Epoch 6, Loss: 1.2534
Epoch 7, Loss: 1.2024
Epoch 8, Loss: 1.1023
Epoch 9, Loss: 1.0922
Epoch 10, Loss: 1.0593
Accuracy: 0.5746
